# Access Keys 

I tried to use S3 credential access to my dataset that is currrently stored into an AWS S3 bucket, however the free verison of databricks prevents credential reads from S3, so therefore I am loading the dataset locally into this notebook and unforunately am reading it from the github repo itself. 

In [0]:
# Replace with your actual keys
#AWS_ACCESS_KEY = "************************"
#AWS_SECRET_KEY = "************************"
#BUCKET = "mlb-career-pipeline-ds-997099833889-us-east-1-an"

#spark.conf.set(f"fs.s3a.access.key", AWS_ACCESS_KEY)
#spark.conf.set(f"fs.s3a.secret.key", AWS_SECRET_KEY)
#spark.conf.set(f"fs.s3a.endpoint", "s3.amazonaws.com")

---------------------------------------------------------------------------
Py4JError                                 Traceback (most recent call last)
File <command-7612748481160445>, line 8
      5 # URL encode the secret key in case it has special characters
      6 encoded_secret = AWS_SECRET_KEY.replace("/", "%2F")
----> 8 dbutils.fs.mount(
      9     source=f"s3a://{AWS_ACCESS_KEY}:{encoded_secret}@{BUCKET}",
     10     mount_point="/mnt/mlb-data"
     11 )
     13 print("Mounted successfully!")
     14 dbutils.fs.ls("/mnt/mlb-data")

File /databricks/python_shell/lib/dbruntime/remotefshandler/RemoteFsHandler.py:48, in prettify_exception_message.<locals>.f_with_exception_handling(*args, **kwargs)
     45 @functools.wraps(f)
     46 def f_with_exception_handling(*args, **kwargs):
     47     try:
---> 48         return f(*args, **kwargs)
     49     except SparkConnectGrpcException as e:
     51         class ExecutionError(Exception):

File /databricks/python_shell/lib/dbruntim

# Bronze Layer: Read CSV
Ideally from S3, instead using a the github repo raw file as a workaround. Commmuinity edition limitations force this github workaround.  

In [0]:
import pandas as pd

URL = "https://raw.githubusercontent.com/a-bishop25/mlb_career_pipeline-DS4DS-/refs/heads/main/sdm2_cum_inj.csv"

# Read via pandas then convert to Spark
pandas_df = pd.read_csv(URL)
bronze_df = spark.createDataFrame(pandas_df)

print(f"Row count: {bronze_df.count()}")
bronze_df.printSchema()

Row count: 63956
root
 |-- Unnamed: 0: long (nullable = true)
 |-- playerID: string (nullable = true)
 |-- yearID: long (nullable = true)
 |-- age: double (nullable = true)
 |-- nameFirst: string (nullable = true)
 |-- nameLast: string (nullable = true)
 |-- bats: string (nullable = true)
 |-- throws: string (nullable = true)
 |-- birthYear: double (nullable = true)
 |-- G: long (nullable = true)
 |-- AB: long (nullable = true)
 |-- H: long (nullable = true)
 |-- X2B: long (nullable = true)
 |-- X3B: long (nullable = true)
 |-- HR: long (nullable = true)
 |-- BB: long (nullable = true)
 |-- HBP: long (nullable = true)
 |-- SF: long (nullable = true)
 |-- SH: long (nullable = true)
 |-- IBB: long (nullable = true)
 |-- PA: long (nullable = true)
 |-- X1B: long (nullable = true)
 |-- wOBA: double (nullable = true)
 |-- years_in_league: long (nullable = true)
 |-- start: long (nullable = true)
 |-- stop: long (nullable = true)
 |-- lag_wOBA: double (nullable = true)
 |-- lag_PA: double (n

# Silver Layer: Clean Data Types

In [0]:
from pyspark.sql.functions import col

# Drop R's row index and rows missing critical columns
silver_df = bronze_df.drop("Unnamed: 0")
silver_df = silver_df.dropna(subset=["age", "wOBA", "years_left", "career_PA", "career_G"])

silver_df = silver_df.select(
    col("playerID"),
    col("nameFirst"),
    col("nameLast"),
    col("yearID").cast("integer"),
    col("age").cast("double"),
    col("wOBA").cast("double"),
    col("lag_wOBA").cast("double"),
    col("lag_PA").cast("double"),
    col("career_PA").cast("integer"),
    col("career_G").cast("integer"),
    col("years_in_league").cast("double"),
    col("total_injuries").cast("double"),
    col("avg_severity").cast("double"),
    col("career_injuries").cast("double"),
    col("career_severity").cast("double"),
    col("missed_prev_year").cast("double"),
    col("years_left").cast("double")
)

print(f"Silver row count: {silver_df.count()}")
silver_df.printSchema()

Silver row count: 60591
root
 |-- playerID: string (nullable = true)
 |-- nameFirst: string (nullable = true)
 |-- nameLast: string (nullable = true)
 |-- yearID: integer (nullable = true)
 |-- age: double (nullable = true)
 |-- wOBA: double (nullable = true)
 |-- lag_wOBA: double (nullable = true)
 |-- lag_PA: double (nullable = true)
 |-- career_PA: integer (nullable = true)
 |-- career_G: integer (nullable = true)
 |-- years_in_league: double (nullable = true)
 |-- total_injuries: double (nullable = true)
 |-- avg_severity: double (nullable = true)
 |-- career_injuries: double (nullable = true)
 |-- career_severity: double (nullable = true)
 |-- missed_prev_year: double (nullable = true)
 |-- years_left: double (nullable = true)



# Write Silver to Delta

Free version is also blocking Databricks DBFS writes as well. 

In [0]:
#silver_df.write.format("delta").mode("overwrite").save("/tmp/mlb_silver")
#print("Silver table written!")

In [0]:
# Community Edition blocks DBFS writes, use in-memory temp views instead
silver_df.createOrReplaceTempView("mlb_silver")
print(f"Silver layer ready: {silver_df.count()} rows")

Silver layer ready: 60591 rows


# Gold Layer
(Dropping NA's)

In [0]:
gold_df = silver_df.dropna()
gold_df.createOrReplaceTempView("mlb_gold")
print(f"Gold row count: {gold_df.count()}")

Gold row count: 47991


In [0]:
import pandas as pd

gold_pdf = gold_with_names.toPandas()
gold_pdf.to_csv("/tmp/mlb_gold_named.csv", index=False)
print(f"Exported {len(gold_pdf)} rows")
print(gold_pdf.head())

Exported 60591 rows
    playerID nameFirst  ... missed_prev_year  years_left
0  abercda01     frank  ...                0           1
1   addybo01       bob  ...                0           7
2  allisar01       art  ...                0           6
3  allisdo01      doug  ...                0          13
4  ansonca01       cap  ...                0          27

[5 rows x 17 columns]


# Converting to Pandas and Prepping Features

In [0]:
import pandas as pd
import numpy as np

# Convert gold layer to pandas for sklearn
pdf = gold_df.toPandas()

# Define features and target
FEATURES = [
    "age", "wOBA", "lag_wOBA", "lag_PA", "career_PA", 
    "career_G", "years_in_league", "total_injuries", 
    "avg_severity", "career_injuries", "career_severity", 
    "missed_prev_year"
]

X = pdf[FEATURES]
y = pdf["years_left"]

print(f"Training set: {X.shape}")
print(f"Target range: {y.min()} to {y.max()} years")

Training set: (47991, 12)
Target range: 1.0 to 32.0 years


# Training with MLFlow

In [0]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train and log with MLflow
mlflow.set_experiment("/Users/a.bishop25@ncf.edu/mlb-career-longevity")

with mlflow.start_run(run_name="random_forest_baseline"):
    model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
    model.fit(X_train, y_train)
    
    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    
    # Log params and metrics
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 10)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)
    
    # Log the model
    mlflow.sklearn.log_model(model, "random_forest_model")
    
    print(f"MAE: {mae:.2f} years")
    print(f"R2 Score: {r2:.3f}")
    print("Model logged to MLflow!")

2026/05/11 02:14:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-c1d301bd-0c17.cloud.databricks.com/ml/experiments/3595170063859469/models/m-0891a061491b4b7b9fced3d9db863f6d?o=7474649107252163
2026/05/11 02:14:28 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when logging the model to auto infer the model signature. To manually set the signature, please visit https://www.mlflow.org/docs/3.8.1/ml/model/signatures.html for instructions on setting signature on models.


MAE: 2.53 years
R2 Score: 0.363
Model logged to MLflow!


# Export for UI Serving

In [0]:
# Model Size check
import base64

with open(f"{local_path}/model.pkl", "rb") as f:
    model_bytes = f.read()

model_b64 = base64.b64encode(model_bytes).decode("utf-8")
print(f"Model size: {len(model_bytes) / 1024 / 1024:.2f} MB")
print(f"Base64 length: {len(model_b64)}")

Model size: 11.07 MB
Base64 length: 15483004


In [0]:
# MODEL EXPORT

import boto3
import io

# Your AWS credentials
AWS_ACCESS_KEY = "XXXXXXXXXXXXXXXXXXXX"
AWS_SECRET_KEY = "XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX"
BUCKET = "mlb-career-pipeline-ds-997099833889-us-east-1-an"

# Upload directly from driver node to S3
s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name="us-east-1"
)

with open(f"{local_path}/model.pkl", "rb") as f:
    s3.upload_fileobj(f, BUCKET, "model/mlb_model.pkl")

print("model.pkl uploaded to S3 successfully!")
print(f"S3 path: s3://{BUCKET}/model/mlb_model.pkl")

model.pkl uploaded to S3 successfully!
S3 path: s3://mlb-career-pipeline-ds-997099833889-us-east-1-an/model/mlb_model.pkl


In [0]:
# DATA EXPORT

import boto3
import pandas as pd

gold_pdf = gold_df.toPandas()
gold_pdf.to_csv("/tmp/mlb_gold_named.csv", index=False)

s3 = boto3.client(
    "s3",
    aws_access_key_id="XXXXXXXXXXXXXXXXXXXX",
    aws_secret_access_key="XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX",
    region_name="us-east-1"
)

s3.upload_file(
    "/tmp/mlb_gold_named.csv",
    "mlb-career-pipeline-ds-997099833889-us-east-1-an",
    "data/mlb_gold_named.csv"
)

print(f"Exported {len(gold_pdf)} rows to S3!")

Exported 47991 rows to S3!
